# Criação de origem_destino_customedio.csv

Esse notebook é utilizado para carregar os dados brutos e criar a tabela dados_processados/origem_destino_customedio.csv. Essa tabela nos informa o preço médio cobrado, por cada compania aérea, para cada Origem -> Destino.

In [26]:
import glob
import pandas as pd 
import numpy as np  

A base de dados nos dá informações sobre quanto foi pago em tarifas aéreas no ano de 2026. A base de dados está imcompleta, mas já tem dados até setembro. Esse código lê as tabelas sobre tarifas aéreas de cada mês disponível em 2026:

In [27]:
# sintaxe de generator function. funciona na base da fé!
df = pd.concat(
    (pd.read_csv(f, sep=';', encoding='latin-1') for f in glob.glob('dados_brutos/*.csv')),
    ignore_index=True,
)
df.sample(5)

,nr_ano_referencia,nr_mes_referencia,sg_empresa_icao,sg_icao_origem,sg_icao_destino,nr_tarifa,nr_assentos,Unnamed: 0,origem,destino,custo_por_assento
3513199,2026.0,5.0,GLO,SBBR,SBSP,"372,9",6.0,NaN,NaN,NaN,NaN
5518161,2026.0,7.0,TAM,SBEG,SBCT,"1258,91",4.0,NaN,NaN,NaN,NaN
1866366,2026.0,3.0,GLO,SBLO,SBSP,"528,61",4.0,NaN,NaN,NaN,NaN
3304280,2026.0,5.0,AZU,SBPA,SBBR,"969,98",1.0,NaN,NaN,NaN,NaN
4805456,2026.0,6.0,TAM,SBTE,SBJV,"1103,9",5.0,NaN,NaN,NaN,NaN


In [28]:
df.columns

df = df.rename(columns={'nr_ano_referencia': 'ano', 'nr_mes_referencia': 'mes', 'sg_empresa_icao': 'empresa',
                   'sg_icao_origem': 'origem', 'sg_icao_destino': 'destino', 'nr_tarifa': "valor_tarifa"})

Ajustando o tipo da coluna 'valor_tarifa' de str para float

In [29]:
display(df.dtypes)
# O valor da tarifa está como str!
print(df.columns)

def valor_tarifa_para_float(valor: str) -> float:
    valor = valor.replace(",", ".")
    valor = float(valor)
    valor = np.float16(valor)
    return valor

df['valor_tarifa'] = df['valor_tarifa'].apply(valor_tarifa_para_float)

df.dtypes

# Eba!!!

ano                  float64
mes                  float64
empresa                  str
origem                   str
destino                  str
valor_tarifa             str
nr_assentos          float64
Unnamed: 0           float64
origem                   str
destino                  str
custo_por_assento    float64
dtype: object

Index(['ano', 'mes', 'empresa', 'origem', 'destino', 'valor_tarifa',
       'nr_assentos', 'Unnamed: 0', 'origem', 'destino', 'custo_por_assento'],
      dtype='str')


AttributeError: 'float' object has no attribute 'replace'

Verificando dados faltantes (não há células em branco!)

In [ ]:
print(f"Dados faltantes totais:\n{df.isna().sum()}")
# AEEE!!!

Dados faltantes totais:
ano             0
mes             0
empresa         0
origem          0
destino         0
valor_tarifa    0
nr_assentos     0
dtype: int64


In [ ]:
df["custo_por_assento"] = df["valor_tarifa"] / df["nr_assentos"]
df.sample(5)

c:\Users\Diogo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,ano,mes,empresa,origem,destino,valor_tarifa,nr_assentos,custo_por_assento
109915,2026,1,AZU,SBFL,SBKP,621.00,1,621.0000
2759784,2026,4,GLO,SBSP,SBSV,486.75,4,121.6875
380837,2026,1,GLO,SBCT,SBSG,2190.00,1,2190.0000
3799935,2026,5,TAM,SBMA,SBCT,843.00,1,843.0000
1534994,2026,3,AZU,SBFL,SBCH,521.50,1,521.5000


In [ ]:
# reset_index faz ser retornado um df
origem_destino_customedio = df.groupby(["mes", "empresa", "origem", "destino"])["custo_por_assento"].mean().reset_index()
origem_destino_customedio

,mes,empresa,origem,destino,custo_por_assento
0,1,ABJ,SBSV,SDLO,346.126543
1,1,ABJ,SBSV,SNCL,190.978720
2,1,ABJ,SDLO,SBSV,321.578571
3,1,ABJ,SNCL,SBSV,269.143711
4,1,AZU,SBAE,SBAR,498.125000
...,...,...,...,...,...
96122,9,TAM,SBZM,SBSR,541.170455
96123,9,TAM,SBZM,SBSV,541.013141
96124,9,TAM,SBZM,SBTE,771.428571
96125,9,TAM,SBZM,SBUL,624.383333


In [ ]:
origens_destinos_unicos = df.groupby(["origem", "destino"])["custo_por_assento"].mean().reset_index()
origens_destinos_unicos.to_csv(path_or_buf="dados_processados/origens_destinos_unicos.csv", sep=";")

In [ ]:
origem_destino_customedio.to_csv(path_or_buf="dados_processados/origem_destino_customedio.csv", sep=";")